# GEGLU — GELU-Gated Linear Unit

源码导航：[core/ffn/geglu.py](../../../core/ffn/geglu.py) 中的 `GEGLUMLP`。

Chowdhery et al. (2022) 在 *PaLM: Scaling Language Modeling with Pathways* 中采用 GEGLU 作为 FFN 激活。与 SwiGLU 相比，GEGLU 仅将门控激活函数由 SiLU 替换为 GELU，公式如下：

$$\text{GEGLU}(x) = \text{GELU}(x W_{\text{gate}}) \odot (x W_{\text{up}}) \; W_{\text{down}}$$

GELU 在原点附近更加平滑，且不存在 SiLU 的负值下凹区，使得门控信号更稳定。

### 1. 理论推导与 GLU 家族对比

Shazeer (2020) 将 GLU 统一抽象为：

$$\text{GLU}(x) = \sigma(x W_g) \odot (x W_u) \; W_d$$

其中 $\sigma$ 为门控激活函数。不同变体仅替换 $\sigma$：

| 变体 | 门控 $\sigma$ | 输出范围 | 特点 |
|---|---|---|---|
| SwiGLU | SiLU | $(-\infty, \infty)$ | 平滑、负值区有微小渗漏 |
| GEGLU | GELU | $(-\infty, \infty)$ | 更平滑、无负值下凹、训练稳定 |
| ReGLU | ReLU | $[0, \infty)$ | 硬截断、计算最简单 |
| GLU | Sigmoid | $(0, 1)$ | 原始 GLU，门控概率化 |

**参数量等价性**：三者均为 3 个权重矩阵（gate / up / down），当 $d_{\text{ffn}} \approx \frac{8}{3} n_{\text{embd}}$ 时，总参数量与标准 4× GELU MLP（2 个矩阵）近似相等。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.ffn.geglu import GEGLUMLP
from core.ffn.swiglu import SwiGLUMLP

### 2. 形状与 dtype 检查

In [ ]:
torch.manual_seed(0)
mlp = GEGLUMLP(n_embd=128, d_ffn=256, dropout=0.0, bias=False)

x = torch.randn(2, 8, 128)
y = mlp(x)

print(f"输入形状: {tuple(x.shape)}")
print(f"输出形状: {tuple(y.shape)}")
assert x.shape == y.shape, "GEGLU 必须保持输入输出维度一致！"
"参数量:", sum(p.numel() for p in mlp.parameters()))

### 3. 门控信号对比：GELU vs SiLU

在同一输入下，对比 GEGLU 与 SwiGLU 的门控输出分布差异。

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(42)
x = torch.randn(2, 16, 128)

geglu = GEGLUMLP(n_embd=128, d_ffn=256, bias=False)
swiglu = SwiGLUMLP(n_embd=128, d_ffn=256, bias=False)

with torch.no_grad():
    geglug = torch.nn.functional.gelu(geglu.gate_proj(x))
    swiglug = torch.nn.functional.silu(swiglu.gate_proj(x))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(geglug.flatten().numpy(), bins=50, color="blue", alpha=0.7)
axes[0].set_title("GEGLU gate output distribution")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Frequency")
axes[0].axvline(0, color="red", linestyle="--")

axes[1].hist(swiglug.flatten().numpy(), bins=50, color="green", alpha=0.7)
axes[1].set_title("SwiGLU gate output distribution")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Frequency")
axes[1].axvline(0, color="red", linestyle="--")

plt.tight_layout()
plt.show()

print(f"GEGLU gate 均值: {geglug.mean().item():.4f},  std: {geglug.std().item():.4f}")
print(f"SwiGLU gate 均值: {swiglug.mean().item():.4f},  std: {swiglug.std().item():.4f}")
print(f"GEGLU gate 负值比例: {(geglug < 0).float().mean().item():.4f}")
print(f"SwiGLU gate 负值比例: {(swiglug < 0).float().mean().item():.4f}")

### 4. 源码精讲

```python
class GEGLUMLP(nn.Module):
    def __init__(self, n_embd, d_ffn, dropout=0.0, bias=False):
        super().__init__()
        self.gate_proj = nn.Linear(n_embd, d_ffn, bias=bias)
        self.up_proj = nn.Linear(n_embd, d_ffn, bias=bias)
        self.down_proj = nn.Linear(d_ffn, n_embd, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        gate = F.gelu(self.gate_proj(x), approximate="tanh")
        up = self.up_proj(x)
        return self.dropout(self.down_proj(gate * up))
```

与 SwiGLU 的唯一区别：第 9 行由 `F.silu` 改为 `F.gelu`。
GELU 的 $\Phi(x)$ 累积分布函数形式使其在原点附近梯度更稳定，门控信号不易出现 SiLU 的剧烈负值下凹。

---

## 延伸阅读与参考资料

### 核心论文
- **GLU Variants Improve Transformer**: Shazeer, 2020. [arXiv:2002.05202](https://arxiv.org/abs/2002.05202)
- **PaLM: Scaling Language Modeling with Pathways**: Chowdhery et al., 2022. [arXiv:2204.02311](https://arxiv.org/abs/2204.02311)

### 相关讨论
- **SwiGLU vs GEGLU**: 两者参数量和计算量完全一致，仅在门控激活上不同。PaLM 使用 GEGLU，LLaMA 使用 SwiGLU，工程上可互换。